# MMFF vs ML-Corrected DFT Pipeline Comparison

SMILES를 입력하면 두 가지 파이프라인을 비교합니다:

| Pipeline | 초기 구조 | 특징 |
|---|---|---|
| **RDKit (MMFF)** | RDKit ETKDG + MMFF 최적화 | 기본값, 빠름 |
| **ML-Corrected** | MMFF + GradientBoosting 결합 길이 보정 | 본 프로젝트 모델 |

**전체 흐름:**
```
SMILES → RDKit ETKDG+MMFF → [선택적 ML 보정] → Gaussian .com → Gaussian 09 → 결과 비교
```

> ⚠️ Gaussian 09W가 설치되어 있어야 하며, `conda activate delta_chem` 환경에서 실행해야 합니다.

## 0. 환경 설정

In [ ]:
import sys, os, math, time
from pathlib import Path

# 프로젝트 루트를 Python path에 추가
project_root = Path("../").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, Image, HTML

# delta_chem 모듈
from delta_chem.ml.corrector import correct_geometry
from delta_chem.chem.gaussian_writer import xyz_to_gaussian_com
from delta_chem.chem.gaussian_runner import run_gaussian
from delta_chem.chem.log_parser import parse_log, parse_final_geometry

MODEL_PATH = str(project_root / "models" / "bond_length_corrector.joblib")
WORK_DIR   = project_root / "data" / "raw" / "notebook_runs"
WORK_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(project_root)  # relative path가 올바르게 동작하도록
print(f"Project root : {project_root}")
print(f"Model path   : {MODEL_PATH}")
print(f"Work dir     : {WORK_DIR}")
print("\n✓ 환경 설정 완료")

## 1. 분석할 SMILES 입력

아래 `SMILES_INPUT`과 `MOL_NAME`을 수정하세요.

In [ ]:
# ──────────────────────────────────────────
#  여기를 수정하세요
# ──────────────────────────────────────────
SMILES_INPUT = "CCO"        # 분석할 SMILES
MOL_NAME     = "ethanol"    # 파일 이름에 쓸 식별자
RUN_GAUSSIAN = True         # False로 설정하면 Gaussian 실행 없이 구조 비교만
# ──────────────────────────────────────────

mol_check = Chem.MolFromSmiles(SMILES_INPUT)
if mol_check is None:
    raise ValueError(f"유효하지 않은 SMILES: {SMILES_INPUT}")

# 2D 구조식 출력
drawer = rdMolDraw2D.MolDraw2DSVG(300, 200)
drawer.DrawMolecule(mol_check)
drawer.FinishDrawing()
svg = drawer.GetDrawingText()
display(HTML(svg))
print(f"분자식: {Chem.rdMolDescriptors.CalcMolFormula(Chem.AddHs(mol_check))}")
print(f"중원자 수: {mol_check.GetNumAtoms()}")
print(f"결합 수: {mol_check.GetNumBonds()}")

## 2. 3D 구조 생성 및 ML 보정

In [ ]:
# ── RDKit MMFF 구조 ──────────────────────────────────────────
mol_mmff = Chem.MolFromSmiles(SMILES_INPUT)
mol_mmff = Chem.AddHs(mol_mmff)
result   = AllChem.EmbedMolecule(mol_mmff, AllChem.ETKDGv3())
if result != 0:
    raise RuntimeError("3D 좌표 생성 실패 (ETKDGv3)")
AllChem.MMFFOptimizeMolecule(mol_mmff)
print("✓ MMFF 구조 생성 완료")

# ── ML 보정 구조 ─────────────────────────────────────────────
mol_ml = correct_geometry(mol_mmff, MODEL_PATH)
print("✓ ML 보정 구조 생성 완료")

# ── 결합 길이 비교표 ──────────────────────────────────────────
def get_bond_lengths(mol):
    conf = mol.GetConformer()
    rows = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ai, aj = mol.GetAtomWithIdx(i), mol.GetAtomWithIdx(j)
        pi, pj = conf.GetAtomPosition(i), conf.GetAtomPosition(j)
        length = math.sqrt((pi.x-pj.x)**2+(pi.y-pj.y)**2+(pi.z-pj.z)**2)
        bt_map = {1.0:"single",1.5:"aromatic",2.0:"double",3.0:"triple"}
        btype  = bt_map.get(bond.GetBondTypeAsDouble(), "other")
        rows.append({
            "bond_idx": bond.GetIdx(),
            "atom1": f"{ai.GetSymbol()}{i}",
            "atom2": f"{aj.GetSymbol()}{j}",
            "bond_type": btype,
            "length_A": round(length, 5),
        })
    return pd.DataFrame(rows)

df_mmff = get_bond_lengths(mol_mmff).rename(columns={"length_A": "mmff_A"})
df_ml   = get_bond_lengths(mol_ml  ).rename(columns={"length_A": "ml_A"})
df_bonds = df_mmff.merge(df_ml[["bond_idx","ml_A"]], on="bond_idx")
df_bonds["delta_A"] = (df_bonds["ml_A"] - df_bonds["mmff_A"]).round(5)

display(df_bonds.style
        .background_gradient(subset=["delta_A"], cmap="RdBu_r", vmin=-0.02, vmax=0.02)
        .format({"mmff_A":"{:.5f}", "ml_A":"{:.5f}", "delta_A":"{:+.5f}"})
        .set_caption("결합 길이: MMFF vs ML-Corrected (Å)"))

## 3. 결합 길이 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"{MOL_NAME}  ({SMILES_INPUT})", fontsize=12, fontweight="bold")

# ── 왼쪽: MMFF vs ML 산점도 ──────────────────────────────────
ax = axes[0]
for btype, grp in df_bonds.groupby("bond_type"):
    ax.scatter(grp["mmff_A"], grp["ml_A"], label=btype, s=60, zorder=3)
    for _, row in grp.iterrows():
        ax.annotate(f"{row['atom1']}-{row['atom2']}",
                    (row["mmff_A"], row["ml_A"]),
                    textcoords="offset points", xytext=(4, 3), fontsize=7)
lims = [df_bonds[["mmff_A","ml_A"]].min().min() - 0.01,
        df_bonds[["mmff_A","ml_A"]].max().max() + 0.01]
ax.plot(lims, lims, "k--", lw=0.8, label="y = x")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("MMFF bond length (Å)")
ax.set_ylabel("ML-corrected bond length (Å)")
ax.set_title("MMFF vs ML-Corrected")
ax.legend(fontsize=8)
ax.set_aspect("equal")

# ── 오른쪽: delta 분포 (보정량 막대) ────────────────────────
ax2 = axes[1]
colors = plt.cm.RdBu_r(
    (df_bonds["delta_A"] - df_bonds["delta_A"].min()) /
    (df_bonds["delta_A"].max() - df_bonds["delta_A"].min() + 1e-9)
)
labels = df_bonds["atom1"] + "-" + df_bonds["atom2"]
ax2.barh(labels.values, df_bonds["delta_A"].values, color=colors, edgecolor="white")
ax2.axvline(0, color="black", lw=0.8, ls="--")
ax2.set_xlabel("ML − MMFF (Å)")
ax2.set_title("Bond Length Correction per Bond")

plt.tight_layout()
plt.savefig(WORK_DIR / f"{MOL_NAME}_structure_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"그래프 저장: {WORK_DIR / f'{MOL_NAME}_structure_comparison.png'}")

## 4. Gaussian 실행 (RUN_GAUSSIAN = True일 때)

두 파이프라인 각각 `.com` 파일을 생성하고 Gaussian을 실행합니다.

In [ ]:
if not RUN_GAUSSIAN:
    print("RUN_GAUSSIAN=False → Gaussian 실행 건너뜀")
else:
    mol_dir = WORK_DIR / MOL_NAME
    mol_dir.mkdir(exist_ok=True)

    def mol_to_xyz(mol, path):
        conf = mol.GetConformer()
        atoms = [mol.GetAtomWithIdx(i) for i in range(mol.GetNumAtoms())]
        with open(path, "w") as f:
            f.write(f"{mol.GetNumAtoms()}\n{MOL_NAME}\n")
            for a in atoms:
                p = conf.GetAtomPosition(a.GetIdx())
                f.write(f"{a.GetSymbol():2s}  {p.x:12.6f}  {p.y:12.6f}  {p.z:12.6f}\n")

    pipelines = {
        "rdkit": mol_mmff,
        "ml":    mol_ml,
    }

    gauss_results = {}

    for tag, mol_obj in pipelines.items():
        print(f"\n── {tag.upper()} 파이프라인 ─────────────────────")
        xyz_p = mol_dir / f"{MOL_NAME}_{tag}.xyz"
        com_p = mol_dir / f"{MOL_NAME}_{tag}.com"

        mol_to_xyz(mol_obj, str(xyz_p))
        xyz_to_gaussian_com(str(xyz_p), str(com_p), title=f"{MOL_NAME} {tag}")

        print(f"  Gaussian 실행 중... ({com_p.name})")
        t0 = time.time()
        out_str = run_gaussian(str(com_p))
        wall = time.time() - t0

        res = parse_log(out_str)
        coords = parse_final_geometry(out_str)
        gauss_results[tag] = {
            "result":    res,
            "coords":    coords,
            "wall_time": wall,
        }

        status = "✓" if res.normal_termination else "✗ 비수렴"
        print(f"  {status}  steps={res.opt_steps}  cpu={res.cpu_seconds:.1f}s  wall={wall:.0f}s")

    print("\n\n모든 Gaussian 계산 완료")

## 5. Gaussian 결과 비교

In [ ]:
if not RUN_GAUSSIAN:
    print("RUN_GAUSSIAN=False → 이 셀을 건너뜁니다")
else:
    # ── 수치 요약표 ──────────────────────────────────────────────
    rows = []
    for tag, d in gauss_results.items():
        r = d["result"]
        rows.append({
            "pipeline":           tag,
            "converged":          r.normal_termination,
            "opt_steps":          r.opt_steps if r.normal_termination else "FAIL",
            "cpu_time_s":         round(r.cpu_seconds, 1) if r.normal_termination else "FAIL",
            "wall_time_s":        round(d["wall_time"], 1),
        })
    df_summary = pd.DataFrame(rows).set_index("pipeline")
    print("=== Gaussian 최적화 결과 요약 ===")
    display(df_summary)

    # ── 스텝 수 / CPU 시간 막대 그래프 ───────────────────────────
    conv = {t: d for t, d in gauss_results.items() if d["result"].normal_termination}
    if len(conv) < 2:
        print("\n(수렴한 파이프라인이 2개 미만 → 비교 그래프 생략)")
    else:
        tags   = list(conv.keys())
        steps  = [conv[t]["result"].opt_steps  for t in tags]
        cpus   = [conv[t]["result"].cpu_seconds for t in tags]
        colors = ["#5B9BD5", "#ED7D31"]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
        fig.suptitle(f"{MOL_NAME}: RDKit vs ML-Corrected", fontweight="bold")

        ax1.bar(tags, steps, color=colors, width=0.4, edgecolor="white")
        ax1.set_ylabel("Optimization Steps")
        ax1.set_title("Steps")
        ax1.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        for x, s in enumerate(steps):
            ax1.text(x, s + 0.1, str(s), ha="center", fontsize=10, fontweight="bold")

        ax2.bar(tags, cpus, color=colors, width=0.4, edgecolor="white")
        ax2.set_ylabel("CPU Time (s)")
        ax2.set_title("CPU Time")
        for x, c in enumerate(cpus):
            ax2.text(x, c + 0.3, f"{c:.1f}s", ha="center", fontsize=10)

        # 개선율 표시
        if len(steps) == 2:
            step_imp = (1 - steps[1]/steps[0]) * 100 if steps[0] > 0 else 0
            cpu_imp  = (1 - cpus[1]/cpus[0])  * 100 if cpus[0]  > 0 else 0
            ax1.set_xlabel(f"ML 개선: {step_imp:+.0f}%")
            ax2.set_xlabel(f"ML 개선: {cpu_imp:+.0f}%")

        plt.tight_layout()
        save_p = WORK_DIR / f"{MOL_NAME}_gaussian_comparison.png"
        plt.savefig(save_p, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"그래프 저장: {save_p}")

## 6. DFT 최적화 후 결합 길이 비교 (MMFF vs ML vs DFT)

In [ ]:
if not RUN_GAUSSIAN:
    print("RUN_GAUSSIAN=False → 이 셀을 건너뜁니다")
else:
    # 두 파이프라인이 모두 수렴했을 때 DFT 결합 길이 비교
    conv = {t: d for t, d in gauss_results.items() if d["result"].normal_termination}
    if len(conv) == 0:
        print("수렴한 결과가 없습니다.")
    else:
        ref_tag    = "ml" if "ml" in conv else list(conv.keys())[0]
        dft_coords = conv[ref_tag]["coords"]

        if len(dft_coords) == 0:
            print("DFT 좌표 파싱 실패")
        else:
            # DFT 결합 길이 계산 (원자 순서 = com 파일 작성 순서 = mol 원자 순서)
            def dft_bond_lengths(bonds_df, coords):
                coord_arr = np.array([(c[1],c[2],c[3]) for c in coords])
                lengths = []
                for _, row in bonds_df.iterrows():
                    # atom1 = e.g. "C0" → index 0
                    i = int(row["atom1"][1:]) if row["atom1"][1:].isdigit() else \
                        int(''.join(filter(str.isdigit, row["atom1"])))
                    j = int(row["atom2"][1:]) if row["atom2"][1:].isdigit() else \
                        int(''.join(filter(str.isdigit, row["atom2"])))
                    if i < len(coord_arr) and j < len(coord_arr):
                        d = np.linalg.norm(coord_arr[i] - coord_arr[j])
                        lengths.append(round(d, 5))
                    else:
                        lengths.append(None)
                return lengths

            df_bonds["dft_A"] = dft_bond_lengths(df_bonds, dft_coords)
            df_bonds["mmff_err"] = (df_bonds["mmff_A"] - df_bonds["dft_A"]).round(5)
            df_bonds["ml_err"]   = (df_bonds["ml_A"]   - df_bonds["dft_A"]).round(5)

            display(df_bonds[["atom1","atom2","bond_type",
                               "mmff_A","ml_A","dft_A",
                               "mmff_err","ml_err"]]
                    .style
                    .background_gradient(subset=["mmff_err","ml_err"], cmap="RdBu_r",
                                         vmin=-0.02, vmax=0.02)
                    .format({c: "{:.5f}" for c in ["mmff_A","ml_A","dft_A","mmff_err","ml_err"]})
                    .set_caption("결합 길이 비교: MMFF / ML / DFT (Å)"))

            mae_mmff = df_bonds["mmff_err"].abs().mean()
            mae_ml   = df_bonds["ml_err"].abs().mean()
            print(f"\n평균 절대 오차 (vs DFT):")
            print(f"  MMFF : {mae_mmff:.5f} Å")
            print(f"  ML   : {mae_ml:.5f} Å")
            if mae_mmff > 0:
                print(f"  ML이 {(1-mae_ml/mae_mmff)*100:.1f}% 더 정확")

## 7. 전체 결과 CSV 저장

In [ ]:
# ── 결합 길이 데이터 저장 ─────────────────────────────────────
bond_csv = WORK_DIR / f"{MOL_NAME}_bond_lengths.csv"
df_bonds.to_csv(bond_csv, index=False)
print(f"결합 길이 CSV: {bond_csv}")

if RUN_GAUSSIAN:
    # ── Gaussian 결과 요약 저장 ──────────────────────────────────
    summary_rows = []
    for tag, d in gauss_results.items():
        r = d["result"]
        summary_rows.append({
            "molecule":    MOL_NAME,
            "smiles":      SMILES_INPUT,
            "pipeline":    tag,
            "converged":   r.normal_termination,
            "opt_steps":   r.opt_steps,
            "cpu_time_s":  round(r.cpu_seconds, 1),
            "wall_time_s": round(d["wall_time"], 1),
        })
    df_gauss = pd.DataFrame(summary_rows)
    gauss_csv = WORK_DIR / f"{MOL_NAME}_gaussian_summary.csv"
    df_gauss.to_csv(gauss_csv, index=False)
    print(f"Gaussian 요약 CSV: {gauss_csv}")
    display(df_gauss)

print("\n✓ 모든 결과 저장 완료")

---
## 참고: 여러 분자 한꺼번에 비교하기

아래 셀을 실행하면 SMILES 리스트 전체를 한 번에 처리하고  
결과를 하나의 CSV에 합쳐서 저장합니다.  
> ⚠️ Gaussian을 각 분자당 2회 실행하므로 시간이 많이 걸립니다.

In [ ]:
# ──────────────────────────────────────────────────────────────
#  원하는 분자를 리스트에 추가하세요
# ──────────────────────────────────────────────────────────────
BATCH_MOLECULES = [
    # (SMILES,           name)
    ("CCO",             "ethanol"),
    ("c1ccccc1",        "benzene"),
    ("CC(=O)O",         "acetic_acid"),
    ("C1CCCCC1",        "cyclohexane"),
    ("CC#N",            "acetonitrile"),
]
RUN_BATCH = False   # True로 바꾸면 실행
# ──────────────────────────────────────────────────────────────

if not RUN_BATCH:
    print("RUN_BATCH=False → 건너뜀")
else:
    batch_rows = []

    for smi, name in BATCH_MOLECULES:
        print(f"\n처리 중: {name} ({smi})")
        mol_dir = WORK_DIR / name
        mol_dir.mkdir(exist_ok=True)

        mol_b = Chem.AddHs(Chem.MolFromSmiles(smi))
        AllChem.EmbedMolecule(mol_b, AllChem.ETKDGv3())
        AllChem.MMFFOptimizeMolecule(mol_b)
        mol_b_ml = correct_geometry(mol_b, MODEL_PATH)

        for tag, m in [("rdkit", mol_b), ("ml", mol_b_ml)]:
            xyz_p = mol_dir / f"{name}_{tag}.xyz"
            com_p = mol_dir / f"{name}_{tag}.com"
            conf  = m.GetConformer()
            with open(xyz_p, "w") as f:
                f.write(f"{m.GetNumAtoms()}\n{name}\n")
                for a in [m.GetAtomWithIdx(i) for i in range(m.GetNumAtoms())]:
                    p = conf.GetAtomPosition(a.GetIdx())
                    f.write(f"{a.GetSymbol():2s}  {p.x:12.6f}  {p.y:12.6f}  {p.z:12.6f}\n")
            xyz_to_gaussian_com(str(xyz_p), str(com_p), title=f"{name} {tag}")
            t0  = time.time()
            out = run_gaussian(str(com_p))
            res = parse_log(out)
            batch_rows.append({
                "molecule":   name,
                "smiles":     smi,
                "pipeline":   tag,
                "converged":  res.normal_termination,
                "opt_steps":  res.opt_steps,
                "cpu_time_s": round(res.cpu_seconds, 1),
                "wall_time_s":round(time.time()-t0, 1),
            })
            st = "✓" if res.normal_termination else "✗"
            print(f"  {tag:6s} {st} steps={res.opt_steps} cpu={res.cpu_seconds:.1f}s")

    df_batch = pd.DataFrame(batch_rows)
    batch_csv = WORK_DIR / "batch_comparison.csv"
    df_batch.to_csv(batch_csv, index=False)
    print(f"\n배치 결과 저장: {batch_csv}")
    display(df_batch)